# SOP-Bench / aircraft_inspection — Qwen3.5-9B trên Kaggle

Notebook riêng để so với Qwen2-7B-Instruct: cùng 112 labeled cases, cùng SOP,
protocol json_transport_v2, cùng mock tools và rubric 2409 v1. Không gồm 22 test
LLM4IAS, không phải code gốc hay leaderboard chính thức SOP-Bench.

**Dùng phiên Kaggle mới, Internet ON, ưu tiên GPU T4 x2.** Không cài thư viện này
vào session Qwen2 đang chạy. Không cần API key hoặc Bedrock.

Default: Qwen/Qwen3.5-9B, FP16, non-thinking, greedy, max_new_tokens=384,
max_turns=12. Đây là controlled baseline do mình chọn, không phải bộ sampling
settings tối ưu do Qwen khuyến nghị. So sánh kết quả/cost theo cấu hình đã khai báo.

Qwen3.5 có kiến trúc hybrid attention khác Qwen2; T4 có thể dùng PyTorch fallback
chậm hơn và tốn RAM hơn khi thiếu kernel tối ưu. Chưa có kết quả inference GPU
xác nhận từ notebook này. Không cài FlashAttention/FLA ngẫu nhiên để sửa tốc độ.

Chạy từng cell: 1 task → 10 task → 112 task. Nếu OOM, giữ log, restart session
trước khi thử NF4; không đổi precision trong một run. Tool là CSV-backed mocks,
không phải simulator vật lý; final-field match không chứng minh SOP compliance.

Nguồn: [model](https://huggingface.co/Qwen/Qwen3.5-9B),
[Transformers](https://huggingface.co/docs/transformers/model_doc/qwen3_5),
[dataset](https://huggingface.co/datasets/amazon/sop-bench).


## 1. Cài môi trường riêng Qwen3.5
Dùng notebook/session mới. Transformers 5.21.1 và Accelerate 1.15.0 đã được
kiểm tra có bản phát hành công khai; không dùng Transformers 4.51.3 của Qwen2.
Giữ torch/CUDA của Kaggle, không cài lại. Nếu đã import HF trong kernel, restart
trước cell này. Chọn Python >=3.11. Dependency warnings cần đọc riêng;
returncode=0 không bảo đảm mọi import/model hoạt động, cell 2 sẽ kiểm tra lớp model.


In [ ]:
import subprocess, sys
if sys.version_info < (3, 11):
    raise RuntimeError("Notebook này cần Python >=3.11; kiểm tra Kaggle environment.")
if "transformers" in sys.modules or "huggingface_hub" in sys.modules:
    raise RuntimeError("Đã import HF trong kernel này: restart session trước khi cài môi trường Qwen3.5.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==5.21.1", "accelerate==1.15.0", "huggingface_hub>=1.31,<3",
    "safetensors>=0.4.3", "pandas", "python-dateutil"], check=True)
print("Install complete. Next: cell 2. Do not install the old Qwen2 requirements here.")


## 2. Cấu hình và GPU
Qwen3.5-9B mặc định FP16, ưu tiên 2 T4, thinking=False. Ngưỡng bộ nhớ chỉ
là kiểm tra ban đầu, không bảo đảm không OOM lúc sinh. Nếu chỉ có một GPU
16 GB, có thể thử LOAD_IN_4BIT=True sau restart, báo cáo NF4 riêng.
Các hyperparameter giữ cùng Qwen2 cho controlled baseline, không tối ưu riêng model mới.


In [ ]:
import os, json, hashlib, platform, uuid, time, csv, shutil
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata
import torch
os.environ["HF_HUB_DISABLE_XET"] = "1"
MODEL_ID = "Qwen/Qwen3.5-9B"
MODEL_SHA = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
LOAD_IN_4BIT = False
ENABLE_THINKING = False  # Fixed non-thinking baseline; do not toggle inside a run.
SEED = 42
MAX_NEW_TOKENS = 384
MAX_INPUT_TOKENS = 16384
MAX_TURNS = 12
WORK = Path("/kaggle/working/sop_aircraft_qwen35_9b")
DATA_DIR = WORK / "author_artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "Bật GPU và Internet; notebook này không dùng TPU."
GPU_INFO = [{"index": i, "name": torch.cuda.get_device_name(i),
    "free_GiB": round(torch.cuda.mem_get_info(i)[0] / 2**30, 2)} for i in range(torch.cuda.device_count())]
print(json.dumps(GPU_INFO, indent=2))
if not LOAD_IN_4BIT and sum(g["free_GiB"] for g in GPU_INFO) < 24:
    raise RuntimeError("Qwen3.5-9B FP16: cần thêm GPU trống, ưu tiên 2 T4; hoặc cấu hình NF4 sau restart.")
if ENABLE_THINKING:
    raise ValueError("Baseline này cố định non-thinking. Thinking cần run/thiết kế ngân sách riêng.")
if LOAD_IN_4BIT:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.45,<1"], check=True)
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration, GenerationConfig, set_seed
print("Transformers:", importlib.metadata.version("transformers"))
print("Model class import OK; precision:", "NF4" if LOAD_IN_4BIT else "FP16", "; thinking=False")


## 3. Adapter được nhúng trong notebook
Chỉ định nghĩa các hàm ở cell này. Không tải dữ liệu, không gọi model ở bước này.
command được parse AST + literal và chỉ dispatch vào whitelist mock tool; không
eval/exec chuỗi do model sinh. finish_inspection là hàm kết thúc của adapter.


In [ ]:
"""Local, model-agnostic adapter for SOP-Bench aircraft_inspection.

No API calls or downloads in this module. Official tools are mock CSV-backed APIs.
The finish_inspection function is our transport adapter, not an original tool.
"""
import ast
import csv
import hashlib
import importlib.util
import inspect
import json
import math
from pathlib import Path
import time

REVISION = "92633eae202fbf66cceb374c6ad55da41ca693b6"
PROTOCOL_VERSION = "json_transport_v2"
EXPECTED_HASHES = {
    "metadata.json": "8ede3a986812c85fea89ba6f56d09fc174142af346d361c80327f3b1933276be",
    "sop.txt": "83e22b9604827ba07b2a1790580a6af1a5a5427d6a80204cdc3303fd47bb941b",
    "tools.py": "571b20ffe2b6ea25e1c3e86798b2f7640efbbe680e05995473f031f5e32171ec",
    "toolspecs.json": "12ae40830defdbe563490852d5d55466d42fbbc895ceae72a82de1141c9809d3",
    "test_set_with_outputs.csv": "20646ee23b8c09a93945eeac0a47bc851bd32564bbf9584ab805e26d3367d0f2",
}


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if hasattr(value, "item"):
        return json_safe(value.item())
    return value


def normalize(value):
    if value is None:
        return None
    value = str(value).strip().lower()
    return None if value in ("", "nan", "none", "null") else value


def prepare(data_dir):
    data_dir = Path(data_dir)
    for name, expected in EXPECTED_HASHES.items():
        assert hashlib.sha256((data_dir / name).read_bytes()).hexdigest() == expected, name
    metadata = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))
    rows = list(csv.DictReader((data_dir / "test_set_with_outputs.csv").open(encoding="utf-8-sig")))
    assert len(rows) == 112 and len({r["aircraft_id"] for r in rows}) == 112
    spec = importlib.util.spec_from_file_location("audited_sop_aircraft_tools", data_dir / "tools.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    manager = module.AircraftInspectionManager()
    toolspecs = json.loads((data_dir / "toolspecs.json").read_text(encoding="utf-8"))
    names = {s["toolSpec"]["name"] for s in toolspecs}
    cases, references = [], {}
    for row in sorted(rows, key=lambda r: r["aircraft_id"]):
        inputs = {k: row[k] if row[k] != "" else None for k in metadata["input_columns"]}
        for key in ("component_weight", "expected_component_weight"):
            if inputs[key] is not None:
                inputs[key] = float(inputs[key])
        cases.append({"case_id": row["aircraft_id"], "inputs": inputs})
        references[row["aircraft_id"]] = {k: row[k] for k in metadata["output_columns"]}
    assert not (set(metadata["input_columns"]) & set(metadata["output_columns"]))
    return cases, references, manager, names, metadata, toolspecs


def execute_command(command, case, manager, names):
    """Only parse literals and dispatch allowlisted mock APIs; never eval/exec model text."""
    if not isinstance(command, str):
        raise ValueError("command must be a string")
    node = ast.parse(command.strip(), mode="eval").body
    if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
        raise ValueError("Expected one named function call")
    if node.args or any(k.arg is None for k in node.keywords):
        raise ValueError("Use explicit keyword arguments, no positional arguments or **kwargs")
    if len({k.arg for k in node.keywords}) != len(node.keywords):
        raise ValueError("Duplicate keyword arguments")
    arguments = {k.arg: ast.literal_eval(k.value) for k in node.keywords}
    name = node.func.id
    if name == "finish_inspection":
        if set(arguments) != {"result"} or not isinstance(arguments["result"], dict):
            raise ValueError("finish_inspection(result={...}) expected")
        return name, arguments, {"finished": True, "result": arguments["result"]}
    if name not in names:
        raise ValueError("Unknown tool: " + name)
    if arguments.get("aircraft_id") != case["case_id"]:
        raise ValueError("Tool must operate on the current aircraft_id")
    inspect.signature(getattr(manager, name)).bind(**arguments)
    observation = manager.process_tool_call(name, arguments)
    return name, arguments, json_safe(observation)


def system_prompt(data_dir, metadata, toolspecs):
    sop = (Path(data_dir) / "sop.txt").read_text(encoding="utf-8")
    return (
        "You are an inspection workflow agent in an offline mock environment. Follow the supplied SOP.\n"
        "Transport protocol for this reconstructed runner: on each turn output ONLY a JSON object "
        "with string keys reason and command. command must contain exactly one allowed Python-style "
        "function call using explicit keyword arguments and literal values. Never invent tool results. "
        "After a tool call, wait for its observation. Do not call tools for other aircraft. "
        "If a tool reports an error, use it as feedback; do not claim it succeeded.\n"
        "For final submission only, replace the SOP's XML reporting wrapper by "
        "finish_inspection(result={...}). Include aircraft_id and these result fields: "
        + json.dumps(metadata["output_columns"]) + ". Use Python literal None for inapplicable fields "
        "inside the command string. This finish function is a runner adapter, not a real operation.\n"
        "Other SOP decision logic is unchanged.\n\nSOP:\n" + sop
        + "\n\nAvailable mock tool specifications:\n" + json.dumps(toolspecs, ensure_ascii=False)
        + transport_reminder(metadata["output_columns"])
    )


def transport_reminder(output_columns):
    example_result = {"aircraft_id": "<current aircraft_id>",
                      **{k: "<value from the corresponding tool observation>" for k in output_columns}}
    example = {"reason": "Summarize the observations supporting this report.",
               "command": "finish_inspection(result=" + repr(example_result) + ")"}
    return (
        "\n\nFINAL TRANSPORT REMINDER (also applies to the last turn):\n"
        "Every response MUST be a JSON object with string reason and command. "
        "A bare function call, XML or Markdown is not a valid response. "
        "finish_inspection(...) must be inside the JSON command STRING, never outside JSON. "
        "Example of the final transport shape only; replace all placeholders using actual observations:\n"
        + json.dumps(example, ensure_ascii=False)
        + "\nInclude all listed result keys, including cross_check_response. "
        "Do not invent success or fill unobserved results with success. "
        "Only use None for genuinely inapplicable fields, not as a substitute for a failed/missing check. "
        "This reminder changes the transport instructions, not the SOP or scoring rules.\n"
    )


def run_case(case, manager, names, sys_prompt, generate, run_dir, max_turns=12):
    run_dir = Path(run_dir)
    result_path = run_dir / "responses" / (case["case_id"] + ".json")
    user_text = "Inspect this aircraft using the SOP and mock tools.\n" + json.dumps(case["inputs"], ensure_ascii=False)
    messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_text}]
    prompt_blob = json.dumps(messages, ensure_ascii=False)
    initial_hash = hashlib.sha256(prompt_blob.encode()).hexdigest()
    if result_path.exists():
        existing = json.loads(result_path.read_text(encoding="utf-8"))
        assert existing["inputs"] == case["inputs"]
        if existing.get("initial_messages_sha256") != initial_hash:
            raise ValueError("Prompt changed: create a new run directory; do not reuse older results.")
        return existing
    record = {"case_id": case["case_id"], "inputs": case["inputs"], "trace": [],
              "finished": False, "final_result": None, "error": None}
    record["initial_messages_sha256"] = initial_hash
    (run_dir / "prompts" / (case["case_id"] + ".json")).write_text(prompt_blob, encoding="utf-8")
    try:
        for turn in range(1, max_turns + 1):
            generated = generate(messages)
            entry = {"turn": turn, **generated, "format_valid": False, "tool_error": None}
            record["trace"].append(entry)
            raw = generated["raw_output"]
            messages.append({"role": "assistant", "content": raw})
            try:
                obj = json.loads(raw.strip())
                if not isinstance(obj, dict) or not all(isinstance(obj.get(k), str) for k in ("reason", "command")):
                    raise ValueError("Expected JSON with string reason and command")
                entry.update(format_valid=True, reason=obj["reason"], command=obj["command"])
                name, arguments, observation = execute_command(obj["command"], case, manager, names)
                entry.update(tool=name, arguments=arguments, observation=observation)
                if name == "finish_inspection":
                    record.update(finished=True, final_result=json_safe(observation["result"]))
                    break
                feedback = {"observation": observation}
            except (ValueError, SyntaxError, TypeError, KeyError) as exc:
                entry["tool_error"] = type(exc).__name__ + ": " + str(exc)
                feedback = {"error": entry["tool_error"]}
            messages.append({"role": "user", "content": "Mock tool feedback: " + json.dumps(feedback, ensure_ascii=False, allow_nan=False)})
    except Exception as exc:
        record["error"] = type(exc).__name__ + ": " + str(exc)
        result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
        raise
    if not record["finished"]:
        record["error"] = "max_turns_reached"
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    return record


def score_result(record, reference, output_columns):
    final = record.get("final_result") or {}
    field_matches = {k: k in final and normalize(final[k]) == normalize(reference[k]) for k in output_columns}
    trace = record["trace"]
    executed = [t.get("tool") for t in trace if t.get("tool") and not t.get("tool_error")]
    required_initial = ["VerifyAircraftClearance", "VerifyMechanicalComponents", "VerifyElectricalSystems"]
    initial_order_ok = executed[:3] == required_initial
    return {"case_id": record["case_id"], "finished": record["finished"],
            "all_fields_match": bool(record["finished"] and all(field_matches.values())),
            "field_matches": field_matches, "initial_three_order_ok": initial_order_ok,
            "valid_format_turns": sum(t["format_valid"] for t in trace), "turns": len(trace),
            "tool_errors": sum(t.get("tool_error") is not None for t in trace),
            "note": "Final-field matching is a supplementary task metric, not LLM4IAS command accuracy or full SOP compliance."}


## 4. Tải dữ liệu, kiểm tra hash và tách input/đáp án
112 là số dòng **có nhãn** thực tế tại revision cố định; không dùng con số 150
trong phần giới thiệu để làm mẫu số. 38 dòng without_outputs không dùng vì thiếu
nhãn công khai cho phép chấm cục bộ. Không đưa các cột đáp án vào prompt.


In [ ]:
from urllib.request import Request, urlopen
for name, expected in EXPECTED_HASHES.items():
    dest = DATA_DIR / name
    if not dest.exists():
        url = f"https://huggingface.co/datasets/amazon/sop-bench/resolve/{REVISION}/data/aircraft_inspection/{name}"
        with urlopen(Request(url, headers={"User-Agent": "SOP-aircraft-research"}), timeout=120) as response:
            blob = response.read()
        assert hashlib.sha256(blob).hexdigest() == expected, f"Hash mismatch: {name}"
        dest.write_bytes(blob)
    assert hashlib.sha256(dest.read_bytes()).hexdigest() == expected, name
    print(name, dest.stat().st_size, "bytes; hash OK")
CASES, REFERENCES, MANAGER, TOOL_NAMES, DATA_METADATA, TOOLSPECS = prepare(DATA_DIR)
SYS_PROMPT = system_prompt(DATA_DIR, DATA_METADATA, TOOLSPECS)
print("Labeled cases:", len(CASES), "Mock tools:", sorted(TOOL_NAMES))
print("First model input (no answers):", json.dumps(CASES[0], ensure_ascii=False, indent=2))


## 5. Tải Qwen3.5-9B và kiểm tra trọng số
Nạp đúng kiến trúc ConditionalGeneration của checkpoint, chỉ cung cấp text.
Không dùng lớp Qwen2 hoặc tự chuyển prefix trọng số. Kiểm tra missing/mismatched
keys và device map trước inference. Vision weights vẫn được nạp dù không dùng ảnh,
nên ngân sách bộ nhớ không hoàn toàn giống model text-only 9B.
Chat template được kiểm tra non-thinking và lưu riêng để kiểm toán.
Cảnh báo thiếu kernel fast path có thể làm chậm; không có nghĩa kết quả đã sai,
nhưng chưa bảo đảm FP16/fallback chạy ổn trên T4 cho mọi task.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=False, trust_remote_code=False)
chat_template = tokenizer.get_chat_template()
assert "enable_thinking" in chat_template, "Unexpected chat template; inspect before inference."
rendered_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Reply with a JSON object."}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
assistant_suffix = rendered_probe.rsplit("<|im_start|>assistant", 1)[-1]
assert "</think>" in assistant_suffix, "Non-thinking prefix not configured; stop to inspect."
kwargs = dict(revision=MODEL_SHA, token=False, trust_remote_code=False,
    dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True,
    attn_implementation="sdpa", ignore_mismatched_sizes=False,
    max_memory={i: int(torch.cuda.mem_get_info(i)[0] * .80) for i in range(torch.cuda.device_count())})
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16)
# Use the published checkpoint's full architecture to avoid text/vision prefix
# conversion errors. Inputs remain text-only: no image tensors are supplied.
model, loading_info = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID, output_loading_info=True, **kwargs)
print("Missing keys:", loading_info.get("missing_keys", []))
print("Unexpected key count:", len(loading_info.get("unexpected_keys", [])))
if loading_info.get("missing_keys") or loading_info.get("mismatched_keys"):
    raise RuntimeError("Missing/mismatched model weights: do not infer with randomly initialized weights.")
model.eval()
DEVICE_MAP = {str(k): str(v) for k,v in model.hf_device_map.items()}
if any(v in ("cpu", "disk") for v in DEVICE_MAP.values()):
    raise RuntimeError("CPU/disk offload detected. Restart with suitable GPUs or NF4; don't time it as pure GPU inference.")
set_seed(SEED)
eos_id = model.generation_config.eos_token_id or tokenizer.eos_token_id
GENERATION = GenerationConfig(do_sample=False, num_beams=1,
    max_new_tokens=MAX_NEW_TOKENS, use_cache=True,
    eos_token_id=eos_id, pad_token_id=tokenizer.pad_token_id)
RUN_DIR = WORK / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_qwen35_9b_" + uuid.uuid4().hex[:6])
for sub in ("responses", "prompts"):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
versions = {n: importlib.metadata.version(n) for n in ["torch", "transformers", "accelerate", "huggingface_hub", "pandas", "safetensors"]}
if LOAD_IN_4BIT: versions["bitsandbytes"] = importlib.metadata.version("bitsandbytes")
metadata = {"dataset": "amazon/sop-bench/aircraft_inspection", "dataset_revision": REVISION,
    "hashes": EXPECTED_HASHES, "model_id": MODEL_ID, "model_sha": MODEL_SHA,
    "model_class": type(model).__name__, "input_modality": "text_only; full checkpoint architecture loaded",
    "precision": "NF4" if LOAD_IN_4BIT else "FP16", "seed": SEED,
    "enable_thinking": False, "max_turns": MAX_TURNS, "max_input_tokens": MAX_INPUT_TOKENS,
    "generation": GENERATION.to_dict(), "gpu": GPU_INFO, "device_map": DEVICE_MAP,
    "versions": versions, "python": platform.python_version(), "cuda": torch.version.cuda,
    "rubric_base": "rubric_2409_v1; accepted 2026-10-29", "protocol_version": PROTOCOL_VERSION,
    "system_prompt_sha256": hashlib.sha256(SYS_PROMPT.encode()).hexdigest(),
    "tokenizer_chat_template_sha256": hashlib.sha256(chat_template.encode()).hexdigest(),
    "loading_info": loading_info,
    "adapter": "Same mock tools, SOP and JSON transport v2 as Qwen2; only model-specific chat wrapping differs",
    "comparison_note": "Greedy non-thinking controlled baseline, not Qwen-recommended optimal sampling; library/tokenizer differences recorded",
    "metric_scope": "Final-field match separate from LLM4IAS command accuracy; C/R manual"}
(RUN_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
(RUN_DIR / "system_prompt.txt").write_text(SYS_PROMPT, encoding="utf-8")
(RUN_DIR / "chat_template.jinja").write_text(chat_template, encoding="utf-8")
(RUN_DIR / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print("Model loaded. Run:", RUN_DIR)
print("Mode: FP16 or declared NF4; thinking=False; greedy; 384 tokens; 12 turns")


## 6. Sinh phản hồi non-thinking
Sử dụng enable_thinking=False trên chat template thật của Qwen3.5.
Không cắt bỏ khối think hoặc sửa JSON ở output để tăng điểm. Nếu model vẫn
sinh văn bản ngoài JSON, giữ raw output và chấm theo rubric F như Qwen2.
Không dùng /nothink làm thay thế cho tham số chat template.


In [ ]:
def generate(messages):
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    batch = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    n_input = batch["input_ids"].shape[-1]
    if n_input > MAX_INPUT_TOKENS:
        raise RuntimeError("Context vượt giới hạn, cần run mới với cấu hình khác; không cắt log tự động.")
    batch = batch.to(model.get_input_embeddings().weight.device)
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    start = time.perf_counter()
    with torch.inference_mode():
        ids = model.generate(**batch, generation_config=GENERATION)[0, n_input:]
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    return {"raw_output": tokenizer.decode(ids, skip_special_tokens=True),
            "input_tokens": n_input, "output_tokens": int(ids.numel()),
            "latency_seconds": time.perf_counter() - start,
            "hit_generation_limit": bool(ids.numel() >= MAX_NEW_TOKENS),
            "rendered_prompt_sha256": hashlib.sha256(rendered.encode()).hexdigest()}


## 7. Smoke test một task
Một task có thể gồm nhiều lượt gọi model và tool. Chạy tối đa 12 lượt; không
retry toàn task để chỉ giữ kết quả đúng. Trong task, phản hồi lỗi tool được đưa
về cho model như observation và vẫn được ghi trong trace.


In [ ]:
first = run_case(CASES[0], MANAGER, TOOL_NAMES, SYS_PROMPT, generate, RUN_DIR, max_turns=MAX_TURNS)
print("Task:", first["case_id"], "finished:", first["finished"], "turns:", len(first["trace"]))
for t in first["trace"]:
    print(t["turn"], t["raw_output"][:700], "ERROR:", t.get("tool_error"))
print("Final:", first["final_result"])
print("Reference (only after inference):", REFERENCES[first["case_id"]])


## 8. Pilot 10 task, sau đó mở rộng 112
Giữ RUN_COUNT=10 cho lần đầu. Sau khi kiểm tra pipeline, đổi thành 112 và chạy lại
cell này, cell 9, cell ZIP. Không cần tải lại model; các task đã lưu được tái sử dụng
với inputs giống nhau. Nếu thay prompt/model/precision/generation thì tạo run mới.
Tất cả task của tập đã chọn phải được giữ trong mẫu số, kể cả hết lượt hoặc sai.


In [ ]:
RUN_COUNT = 10  # Sau pilot, đổi thành 112 để chạy toàn bộ labeled subset.
assert 1 <= RUN_COUNT <= len(CASES)
SELECTED = CASES[:RUN_COUNT]
RESULTS = []
for index, case in enumerate(SELECTED, 1):
    record = run_case(case, MANAGER, TOOL_NAMES, SYS_PROMPT, generate, RUN_DIR, max_turns=MAX_TURNS)
    RESULTS.append(record)
    print(f"{index}/{RUN_COUNT} {case['case_id']} finished={record['finished']} turns={len(record['trace'])} error={record['error']}")


## 9. Chấm task và xuất trace để duyệt theo rubric
Tự động chấm 7 trường kết quả so CSV (không phân biệt hoa thường, thiếu/null
được chuẩn hóa có ghi nhận). Thiếu một key không được tính đúng dù reference null.
`all_fields_match` không chứng minh toàn bộ SOP đúng; kiểm tra thêm thứ tự 3 tool
đầu như chỉ số chẩn đoán, và giữ trace để duyệt hành động theo rubric C/R.
Không cộng hoặc so trực tiếp tỷ lệ này với 4/22 LLM4IAS.


In [ ]:
SCORES = [score_result(r, REFERENCES[r["case_id"]], DATA_METADATA["output_columns"]) for r in RESULTS]
flat = [{"case_id": s["case_id"], "finished": s["finished"], "all_fields_match": s["all_fields_match"],
         "initial_three_order_ok": s["initial_three_order_ok"], "turns": s["turns"],
         "valid_format_turns": s["valid_format_turns"], "tool_errors": s["tool_errors"],
         **{k+"_match": v for k,v in s["field_matches"].items()}} for s in SCORES]
with (RUN_DIR / "task_scores.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=list(flat[0])); writer.writeheader(); writer.writerows(flat)
review = []
for r in RESULTS:
    for t in r["trace"]:
        review.append({"case_id": r["case_id"], "turn": t["turn"], "raw_output": t["raw_output"],
            "format_valid_auto": t["format_valid"], "tool_error": t.get("tool_error"),
            "manual_C_0_1_U": "", "manual_R_1_to_5_U": "", "notes": ""})
with (RUN_DIR / "rubric_review.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=list(review[0])); writer.writeheader(); writer.writerows(review)
summary = {"selected_count": len(RESULTS), "available_labeled_cases": len(CASES),
    "finished": sum(s["finished"] for s in SCORES),
    "all_fields_match": sum(s["all_fields_match"] for s in SCORES),
    "initial_three_order_ok": sum(s["initial_three_order_ok"] for s in SCORES),
    "format_valid_turns": sum(s["valid_format_turns"] for s in SCORES),
    "total_turns": sum(s["turns"] for s in SCORES),
    "tool_errors": sum(s["tool_errors"] for s in SCORES),
    "generate_seconds": sum(t.get("latency_seconds", 0) for r in RESULTS for t in r["trace"]),
    "manual_R": "not scored", "manual_C": "not scored",
    "note": "Supplementary multi-step inspection benchmark; not original LLM4IAS metrics or physical safety verification."}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 10. ZIP kết quả và tải về
Chạy cả khi lỗi giữa chừng để lưu trace. ZIP không chứa trọng số. Đừng đóng
session trước khi tải file hoặc lưu phiên bản/output của Kaggle.


In [ ]:
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Download:", zip_path)
from IPython.display import display, FileLink
display(FileLink(zip_path))
